# 03: Evaluation

In Notebook 02 we ran individual questions by hand. This notebook evaluates the agent
systematically: we upload a dataset subset to Langfuse, run the agent on every item, and
score each response with an LLM-as-judge grader using the official DeepSearchQA methodology.

## What You'll Learn

1. Uploading a DeepSearchQA subset to Langfuse as a persistent dataset
2. The LLM-as-judge grader: precision, recall, F1, and the four outcome categories
3. A single-sample evaluation walkthrough
4. Running the full experiment with `run_experiment`
5. Inspecting and interpreting item-level results

## Prerequisites

Complete Notebooks 01 and 02. You'll need all credentials in `.env`:
- `GOOGLE_API_KEY`
- `LANGFUSE_PUBLIC_KEY` and `LANGFUSE_SECRET_KEY`
- `OPENAI_API_KEY` (for the LLM grader)

In [6]:
import json
import os
import tempfile
from pathlib import Path
from typing import Any

import pandas as pd
from aieng.agent_evals.evaluation import run_experiment
from aieng.agent_evals.evaluation.graders.config import LLMRequestConfig
from aieng.agent_evals.knowledge_qa import DeepSearchQADataset, KnowledgeGroundedAgent
from aieng.agent_evals.knowledge_qa.deepsearchqa_grader import (
    EvaluationOutcome,
    evaluate_deepsearchqa_async,
)
from aieng.agent_evals.knowledge_qa.notebook import display_response, run_with_display
from aieng.agent_evals.langfuse import upload_dataset_to_langfuse
from dotenv import load_dotenv
from IPython.display import HTML, display  # noqa: A004
from langfuse.experiment import Evaluation
from rich.console import Console
from rich.panel import Panel
from rich.table import Table


if Path("").absolute().name == "eval-agents":
    print(f"Working directory: {Path('').absolute()}")
else:
    os.chdir(Path("").absolute().parent.parent)
    print(f"Working directory set to: {Path('').absolute()}")

load_dotenv(verbose=True)
console = Console(width=100)

DATASET_NAME = "DeepSearchQA-Sun-Life"

2026-03-24 15:12:40,808 INFO dotenv.main: python-dotenv could not find configuration file .env.


Working directory set to: /


## 1. Uploading the Dataset to Langfuse

Langfuse stores our evaluation dataset so we can run multiple experiments against the same items
and compare results over time. Each dataset item has three fields:

- **`input`**: the question (sent to the agent)
- **`expected_output`**: the ground truth answer (given to the grader, never shown to the agent)
- **`metadata`**: `category`, `answer_type`, `example_id`

Items are deduplicated by a hash of their content, so running this cell again is safe.

## 1. Collecting Tool Calls and Enriching the Dataset

This notebook enriches the DeepSearchQA dataset by running the agent on each example and capturing
the tool calls it makes. These tool calls become part of the "expected output" that can be used
for evaluation.

The enriched dataset will include:
- **`expected_tool_calls`**: List of tool calls with names and arguments (e.g., google_search, web_fetch)

This allows us to evaluate not just the final answer quality, but also whether the agent is using
the right tools with appropriate arguments.

In [ ]:
# Load the dataset examples
dataset = DeepSearchQADataset()
examples = dataset.get_by_category("Finance & Economics")[:10]

console.print(f"Loaded [cyan]{len(examples)}[/cyan] examples from Finance & Economics category")

In [ ]:
# Step 1: Run agent on all examples and collect tool calls
console.print("[cyan]Running agent on examples to collect tool calls...[/cyan]")

agent = KnowledgeGroundedAgent()
enriched_examples = []

for i, ex in enumerate(examples):
    console.print(f"[yellow]Processing example {i+1}/{len(examples)}[/yellow]: {ex.problem[:80]}...")
    
    # Run agent and get response
    response = agent.answer(ex.problem)
    
    # Add tool calls to example
    ex.expected_tool_calls = response.tool_calls
    enriched_examples.append(ex)
    
    # Reset agent state for next question
    agent.reset()
    
    console.print(f"  [green]✓[/green] Collected {len(response.tool_calls)} tool calls")

console.print(f"[green]✓[/green] Finished collecting tool calls for all examples")

# Step 2: Save enriched dataset to file
output_dir = Path("sunlife")
output_dir.mkdir(exist_ok=True)
output_path = output_dir / "enriched_dataset.jsonl"

console.print(f"[cyan]Saving enriched dataset to {output_path}...[/cyan]")

with open(output_path, "w", encoding="utf-8") as f:
    for ex in enriched_examples:
        record = {
            "example_id": ex.example_id,
            "problem": ex.problem,
            "problem_category": ex.problem_category,
            "answer": ex.answer,
            "answer_type": ex.answer_type,
            "expected_tool_calls": ex.expected_tool_calls,
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

console.print(f"[green]✓[/green] Saved enriched dataset with {len(enriched_examples)} examples")

# Step 3: Upload enriched dataset to Langfuse
console.print(f"[cyan]Uploading enriched dataset to Langfuse...[/cyan]")

with tempfile.NamedTemporaryFile(mode="w", suffix=".jsonl", delete=False, encoding="utf-8") as f:
    for ex in enriched_examples:
        record = {
            "input": ex.problem,
            "expected_output": ex.answer,
            "metadata": {
                "example_id": ex.example_id,
                "category": ex.problem_category,
                "answer_type": ex.answer_type,
                "expected_tool_calls": ex.expected_tool_calls,
            },
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
    temp_path = f.name

await upload_dataset_to_langfuse(dataset_path=temp_path, dataset_name=DATASET_NAME)
os.unlink(temp_path)

console.print(f"[green]✓[/green] Dataset '{DATASET_NAME}' uploaded to Langfuse with tool calls")